In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

In [ ]:
# Load the student dataset
df = pd.read_csv("student_exam_data.csv")

print(df.head())
print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nPass / Fail count:")
print(df["passed"].value_counts())

In [ ]:
# Separate input features and target
X = df[[
    "study_hours",
    "attendance_pct",
    "past_score",
    "practice_tests",
    "sleep_hours"
]]
y = df["passed"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# Train a Decision Tree model
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

In [ ]:
# Train a Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    oob_score=True,
    random_state=42
)
rf_model.fit(X_train, y_train)

print("Number of trees:", len(rf_model.estimators_))
print("Out-of-Bag score:", rf_model.oob_score_)

In [ ]:
# Make predictions and find class probabilities
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)

results = X_test.copy()
results["Actual"] = y_test
results["Predicted"] = rf_pred
results["Probability_Pass"] = rf_prob[:, 1]

display(results.round(3))

In [ ]:
# Evaluate the Random Forest model
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred, zero_division=0))
print("Recall:", recall_score(y_test, rf_pred, zero_division=0))
print("F1-score:", f1_score(y_test, rf_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))

In [ ]:
# Compare Decision Tree and Random Forest accuracy
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

display(comparison)

plt.bar(comparison["Model"], comparison["Accuracy"])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Decision Tree vs Random Forest")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Find and visualize feature importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.barh(importance["Feature"], importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Try different numbers of trees
tree_counts = [10, 25, 50, 100, 200]
scores = []

for n in tree_counts:
    model = RandomForestClassifier(
        n_estimators=n,
        max_depth=3,
        random_state=42
    )
    model.fit(X_train, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test)))

tuning_results = pd.DataFrame({
    "Number of Trees": tree_counts,
    "Test Accuracy": scores
})
display(tuning_results)

plt.plot(tree_counts, scores, marker="o")
plt.xlabel("Number of Trees")
plt.ylabel("Test Accuracy")
plt.title("Effect of Number of Trees")
plt.show()

In [ ]:
# Predict the result for a new student
new_student = pd.DataFrame({
    "study_hours": [6],
    "attendance_pct": [80],
    "past_score": [70],
    "practice_tests": [4],
    "sleep_hours": [7]
})

prediction = rf_model.predict(new_student)
probability = rf_model.predict_proba(new_student)

print("Predicted class:", "Pass" if prediction[0] == 1 else "Fail")
print("Probability of passing:", round(probability[0][1], 3))

## Explain the concepts in your own words

1. **What is Random Forest?**

   Random Forest is a Machine Learning algorithm that combines many decision trees to make a final prediction.

2. **What is ensemble learning?**

   Ensemble learning combines multiple models to get a better and more stable prediction.

3. **What is bagging?**

   Bagging trains different decision trees using different random samples of the training data.

4. **What is feature importance?**

   Feature importance shows how much each input feature helps the model make predictions. It does not show whether the feature increases or decreases the result.

5. **What is the Out-of-Bag (OOB) score?**

   OOB score evaluates the model using training samples that were left out when individual trees were created.

6. **Why do we compare Decision Tree and Random Forest?**

   We compare them to understand whether combining many trees improves prediction performance.

7. **Why do we use `random_state=42`?**

   It helps us get the same random split and reproducible model results when we run the notebook again.